In [2]:
import pandas as pd

In [3]:
df = pd.read_csv("../tudoapenasentidades.csv")
df_sem_suplicata = pd.read_csv("../tudosemduplicatas.csv")

In [4]:
df_sem_suplicata

,timestamp,entity_id,old_state,new_state,attributes
0,2025-07-13T14:00:10.408111,automation.aut_status_bruno,on,on,"{""id"": ""1751855857509"", ""last_triggered"": ""202..."
1,2025-07-13T14:00:15.773224,sensor.arris_tg1692a_router_download_speed,0.0569704745759571,0.0736161792452594,"{""state_class"": ""measurement"", ""unit_of_measur..."
2,2025-07-13T14:00:15.774808,sensor.arris_tg1692a_router_upload_speed,0.167631198829402,0.168477584259541,"{""state_class"": ""measurement"", ""unit_of_measur..."
3,2025-07-13T14:00:40.716334,media_player.dlna_b13_box,unavailable,unknown,"{""friendly_name"": ""DLNA-B13 Box"", ""supported_f..."
4,2025-07-13T14:00:47.630193,camera.192_168_0_33,idle,idle,"{""access_token"": ""fcf5c1a49773eb691c898b67db97..."
...,...,...,...,...,...
280,2025-07-17T16:26:50.351545,automation.auto_sc4,off,on,"{""id"": ""1749219558697"", ""last_triggered"": ""202..."
281,2025-07-17T16:26:55.126648,automation.track_ar_209_state,on,off,"{""id"": ""1748634239642"", ""last_triggered"": ""202..."
282,2025-07-17T16:27:09.070992,scene.knob_long_press_spotlight,unknown,unavailable,"{""restored"": true, ""supported_features"": 0}"
283,2025-07-06T23:23:38.881268,input_button.estou_trabalhando,unknown,unknown,"{""editable"": true, ""icon"": ""mdi:button-pointer..."


### Identificação de Sensores de Presença

> Para identificar sensores de presença no conjunto de dados, foram utilizados os seguintes critérios:

1. **Eliminação de entidades irrelevantes**  
   Remoção de entidades conhecidas que não representam sensores de presença, como: `camera`, `media_player`, `router`, `scene`, `automation`, entre outros.
2. **Filtragem por palavras-chave no `entity_id`**  
   Seleção de entidades com nomes que indicam presença ou movimento, como:  
   `presence`, `motion`, `occupancy`, `presenca`.
3. **Análise de mudanças de estado binário**  
   Identificação de entidades que alternam frequentemente entre `on` e `off`, padrão típico de sensores de presença.
4. **Verificação de metadados nos atributos**  
   Análise da coluna `attributes` em busca de indicadores como:  
   - `"device_class": "motion"` ou `"occupancy"`  
   - `"friendly_name"` contendo termos como `presença`, `movimento` ou `motion`

In [ ]:
import json
filtro_pres = df_sem_suplicata.loc[df_sem_suplicata["entity_id"].str.contains("pres", case=False)]
filtro_pres = filtro_pres[:-1]
filtro_pres

estado = filtro_pres["old_state"]

lista_json = []
for _, row in filtro_pres.iterrows():
    lista_json.append(json.loads(row["attributes"]))


elementos = []
for elemento in lista_json:
    elementos.append(elemento.get("friendly_name"))


df_comb = pd.DataFrame({
    "friendly_name": elementos,
    "old_state": filtro_pres["old_state"]
})


0            Presença sala Motion
1        Presença cozinha  Motion
2        Presença banheiro Motion
3      Presença Quarto  Occupancy
4    Presença Quarto  Sensitivity
dtype: object